<a href="https://colab.research.google.com/github/perfectmakuwerere/efficient-llm-lab/blob/main/SmolLM2_1_7B_W4A16_instruct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
!pip  install llmcompressor

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 710.8/710.8 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 142.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.610.43
    Uninstalling nvidia-ml-py-13.610.43:
      Successfully uninstalled nvidia-ml-py-13.610.43
  Attempting

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MODEL_DIR = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
OUTPUT_DIR = "./SmolLM2-1.7B-W4A16"

print(f"Base model:      {MODEL_DIR}")
print(f"Quantized model: {OUTPUT_DIR}")

Base model:      HuggingFaceTB/SmolLM2-1.7B-Instruct
Quantized model: ./SmolLM2-1.7B-W4A16


In [ ]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


In [ ]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

2026-06-11T16:11:42.4173 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.q_proj using 256 samples
2026-06-11T16:11:43.6615 | GPTQ | METRIC - time 1.24s
2026-06-11T16:11:43.6622 | GPTQ | METRIC - error 14531.08
2026-06-11T16:11:43.6638 | GPTQ | METRIC - Accelerator 0 | usage: 3.36% | total memory: 42.4 Gb
2026-06-11T16:11:43.6737 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.k_proj using 256 samples
2026-06-11T16:11:44.9189 | GPTQ | METRIC - time 1.24s
2026-06-11T16:11:44.9197 | GPTQ | METRIC - error 15312.68
2026-06-11T16:11:44.9214 | GPTQ | METRIC - Accelerator 0 | usage: 3.36% | total memory: 42.4 Gb
2026-06-11T16:11:44.9316 | compress_module_list | INFO - Quantizing model.layers.23.self_attn.v_proj using 256 samples
2026-06-11T16:11:46.1812 | GPTQ | METRIC - time 1.25s
2026-06-11T16:11:46.1819 | GPTQ | METRIC - error 59693.09
2026-06-11T16:11:46.1833 | GPTQ | METRIC - Accelerator 0 | usage: 3.36% | total memory: 42.4 Gb
2026-06-11T16:11:46

(25/25): Propagating: 100%|██████████| 256/256 [00:02<00:00, 116.98it/s]

2026-06-11T16:11:57.2168 | finalize | INFO - Compression lifecycle finalized for 1 modifiers



Dispatching model: 100%|██████████| 319/319 [00:00<00:00, 25604.88it/s]

Quantization complete. Model saved to: ./SmolLM2-1.7B-W4A16


In [ ]:
prompt = "Machine learning is a branch of"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = base_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Base Model ({MODEL_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Base Model (HuggingFaceTB/SmolLM2-1.7B-Instruct)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that involves the use of algorithms and statistical models to enable machines to perform a specific task. It is a subset of AI that focuses on the development of computer programs that can learn from and make decisions or predictions based on data.

Machine learning algorithms can be trained on large datasets to identify


In [ ]:
import logging
logging.getLogger("llmcompressor").setLevel(logging.WARNING)

quant_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, device_map="cpu", dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt")
outputs = quant_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)
generated = outputs[0][inputs["input_ids"].shape[-1]:]

print(f"Quantized Model ({OUTPUT_DIR})")
print(f"Prompt: {prompt}")
print(f"Response: {tokenizer.decode(generated, skip_special_tokens=True)}")

Decompressing model: 100%|██████████| 168/168 [00:04<00:00, 38.49it/s]


Quantized Model (./SmolLM2-1.7B-W4A16)
Prompt: Machine learning is a branch of
Response:  artificial intelligence that uses algorithms to automatically learn and improve from experience. It is a type of machine learning that uses statistical techniques to enable machines to learn from data, without being programmed.

Machine learning is used in a variety of applications, including:

* Image and speech recognition
* Natural


In [ ]:
from datasets import load_dataset

def calculate_perplexity(model, tokenizer, dataset, max_tokens=5000, stride=512):
    encodings = tokenizer(
        "\n\n".join(dataset["text"]),
        return_tensors="pt", truncation=True, max_length=max_tokens,
    )
    input_ids = encodings.input_ids
    nlls, prev_end = [], 0

    for begin_loc in range(0, input_ids.size(1), stride):
        end_loc = min(begin_loc + stride, input_ids.size(1))
        trg_len = end_loc - prev_end
        input_slice = input_ids[:, begin_loc:end_loc]
        target_slice = input_slice.clone()
        target_slice[:, :-trg_len] = -100
        with torch.no_grad():
            loss = model(input_slice, labels=target_slice).loss
            nlls.append(loss * trg_len)
        prev_end = end_loc

    return math.exp(torch.stack(nlls).sum() / prev_end)

test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
print(f"Loaded {len(test_data)} test samples")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Loaded 4358 test samples


In [ ]:
quant_ppl = calculate_perplexity(quant_model, tokenizer, test_data)
print(f"Quantized perplexity: {quant_ppl:.2f}")

Quantized perplexity: 15.37


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, device_map="cpu", dtype=torch.bfloat16,
)
base_ppl = calculate_perplexity(base_model, tokenizer, test_data)
print(f"Base perplexity: {base_ppl:.2f}")

Base perplexity: 13.54


In [ ]:
print("Perplexity Comparison")
print("=" * 40)
print(f"Base (BF16):      {base_ppl:.2f}")
print(f"Quantized (W4A16): {quant_ppl:.2f}")
print(f"Difference:       {quant_ppl - base_ppl:+.2f} ({(quant_ppl/base_ppl - 1)*100:+.1f}%)")
print(f"\nA small increase in perplexity is expected — the quantized layers use 4-bit weights.")

Perplexity Comparison
Base (BF16):      13.54
Quantized (W4A16): 15.37
Difference:       +1.83 (+13.5%)

A small increase in perplexity is expected — the quantized layers use 4-bit weights.
